In [0]:
# imports
import pyspark.sql.functions as F

In [0]:
# gold class
class GoldAggregations:
    def __init__(self, catalog_name, silver_table_schema, silver_table_name, gold_schema, gold_aggregated_table_name, gold_feature_table_name):
        self.catalog_name = catalog_name
        self.silver_table_schema = silver_table_schema
        self.silver_table_name = silver_table_name
        self.gold_schema = gold_schema
        self.gold_aggregated_table_name = gold_aggregated_table_name
        self.gold_feature_table_name = gold_feature_table_name

    def build_gold_aggregated_table(self):
        silver_table_name = self.catalog_name + '.' + self.silver_table_schema + '.' +  self.silver_table_name
        gold_table_name = self.catalog_name + '.' + self.gold_schema + '.' +  self.gold_aggregated_table_name

        print(f"03: Building Gold Aggregated Table, Path: {gold_table_name}")
        spark.sql(f"""
            CREATE OR REPLACE TABLE {gold_table_name} AS
            SELECT 
                transaction_date,
                product_name,
                destination_city,
                ROUND(total_demand_quantity,2) AS total_demand,
                ROUND(avg_unit_price_usd, 2) AS avg_unit_price,
                ROUND(total_available_inventory, 2) AS total_inventory,
                CURRENT_TIMESTAMP() AS gold_ingestion_timestamp
            FROM (
                SELECT 
                    transaction_date,
                    product_name,
                    destination_city,
                    SUM(demand_quantity) AS total_demand_quantity,
                    AVG(unit_price_usd) AS avg_unit_price_usd,
                    SUM(available_inventory) AS total_available_inventory
                FROM {silver_table_name}
                GROUP BY transaction_date, product_name, destination_city
                )
            """)
        print(f"04: Gold Aggregated Table Created")

    def build_gold_feature_table(self):
        gold_aggregated_table = self.catalog_name + '.' + self.gold_schema + '.' +  self.gold_aggregated_table_name
        gold_feature_table_name = self.catalog_name + '.' + self.gold_schema + '.' +  self.gold_feature_table_name

        print(f"05: Creating & loading data to table: {gold_feature_table_name}")
        spark.sql(f"""
            CREATE OR REPLACE TABLE {gold_feature_table_name} AS 
            SELECT 
                transaction_date,
                product_name,
                destination_city,
                total_demand,
                avg_unit_price,
                total_inventory,
                dayofweek(transaction_date) AS day_of_week,
                month(transaction_date) AS month,
                ROUND(LAG(total_demand) OVER (PARTITION BY product_name, destination_city ORDER BY transaction_date), 2) AS demand_lag_1,
                ROUND(LAG(total_demand, 7) OVER (PARTITION BY product_name, destination_city ORDER BY transaction_date), 2) AS demand_lag_7,
                ROUND(AVG(total_demand) OVER (PARTITION BY product_name, destination_city ORDER BY transaction_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS moving_avg_7
            FROM {gold_aggregated_table}    
        """)
        print(f"06: Data loaded, table: {gold_feature_table_name}")

    def read_silver_table(self):
        silver_table_path = self.catalog_name + '.' + self.silver_table_schema + '.' + self.silver_table_name
        print(f"01: Loading the silver table, path {silver_table_path}")
        self.silver_data = spark.read.table(silver_table_path)
        print("02: Silver table loaded")



In [0]:
# main part
gold_aggregations = GoldAggregations(
    catalog_name="supply_chain",
    silver_table_schema="silver",
    silver_table_name="silver_supply_chain",
    gold_schema="gold",
    gold_aggregated_table_name="daily_demand",
    gold_feature_table_name="daily_demand_features"
)

gold_aggregations.read_silver_table()
gold_aggregations.build_gold_aggregated_table()
gold_aggregations.build_gold_feature_table()

In [0]:
%sql
SELECT * 
FROM supply_chain.gold.daily_demand;

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM supply_chain.gold.daily_demand;

In [0]:
%sql
SELECT * 
FROM supply_chain.gold.daily_demand_featuress;

In [0]:
%sql
SELECT COUNT(*) AS rows_count 
FROM supply_chain.gold.daily_demand_featuress;